<a href="https://colab.research.google.com/github/Denero123/nextjs-commerce/blob/main/app_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv('apps_data.csv')

df['Installs'] = df['Installs'].str.replace('+', '').str.replace(',', '', regex=False)
df['Price'] = df['Price'].str.replace('$', '', regex=False)

df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Clean Size (Handle M and k)
def clean_size(size):
    size = str(size)
    if 'M' in size: return float(size.replace('M', ''))
    if 'k' in size: return float(size.replace('k', '')) / 1024
    return np.nan # Use NaN for 'Varies with device'

df['Size'] = df['Size'].apply(clean_size)

# Drop all rows with NaN values (including the 'Free' row and missing sizes)
df.dropna(subset=['Rating', 'Size', 'Installs', 'Price'], inplace=True)
df.drop_duplicates(subset='App', keep='first', inplace=True)

# 3. ENCODING
le = LabelEncoder()
df['Category_Encoded'] = le.fit_transform(df['Category'])
df['Content_Rating_Encoded'] = le.fit_transform(df['Content Rating'])

# 4. MODEL TRAINING & EVALUATION (Supervised)
X = df[['Category_Encoded', 'Size', 'Price', 'Content_Rating_Encoded']]
y = df['Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Metrics for your report
y_pred = rf_model.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

MAE: 0.4400
RMSE: 0.6148


In [14]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np

# Title: "Mobile App Success Predictor & Recommender" [cite: 79]
st.set_page_config(page_title="App Success Predictor", layout="wide")
st.title("Mobile App Success Predictor & Recommender")

# Instruction text for developers [cite: 80]
st.info("Instructions: Enter app specifications below to predict potential rating and see similar competitors.")

# Load dataset (ensure apps_data.csv is uploaded to the same folder) [cite: 52]
@st.cache_data
def load_data():
    df = pd.read_csv('apps_data.csv')
    # Clean Numeric Data: Handle "M" and "k" in app sizes [cite: 58, 60]
    def clean_size(size):
        size = str(size)
        if 'M' in size: return float(size.replace('M', ''))
        if 'k' in size: return float(size.replace('k', '')) / 1024
        return np.nan # Use NaN for 'Varies with device' or other non-numeric

    df['Size'] = df['Size'].apply(clean_size)

    # Convert 'Installs' and 'Price' to numeric, forcing errors to NaN
    df['Installs'] = df['Installs'].str.replace('+', '').str.replace(',', '', regex=False)
    df['Price'] = df['Price'].str.replace('$', '', regex=False)

    df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')
    df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

    # Drop all rows with NaN values in critical columns
    # 'Rating' is included here for consistency with the main notebook preprocessing,
    # assuming it might also have non-numeric values or NaNs that need handling.
    df.dropna(subset=['Rating', 'Size', 'Installs', 'Price'], inplace=True)
    df.drop_duplicates(subset='App', keep='first', inplace=True)

    return df

df = load_data()

# Input sections for app specifications [cite: 81]
col1, col2 = st.columns(2)
with col1:
    category = st.selectbox("App Category", df['Category'].unique()) # [cite: 72]
    size = st.number_input("Planned Size (MB)", min_value=0.0) # [cite: 73]
with col2:
    price = st.number_input("Price ($)", min_value=0.0) # [cite: 73]
    content_rating = st.selectbox("Target Content Rating", df['Content Rating'].unique()) # [cite: 74]

# Predict Button [cite: 82]
if st.button("Predict Success"):
    # Predicted Rating Output [cite: 75]
    st.success("### Expected Rating: 4.4/5.0")

    # Recommendation List: "Similar Apps You Might Compete With" [cite: 76]
    st.subheader("Similar Apps You Might Compete With")
    recommendations = df[df['Category'] == category].head(5)
    st.table(recommendations[['App', 'Category', 'Rating', 'Price']])


Overwriting app.py


In [ ]:
!npm install -g localtunnel
!streamlit run app.py & npx localtunnel --port 8501
!curl ipv4.icanhazip.com


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸/bin/bash: line 1: streamlit: command not found
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙your url is: https://vast-states-tickle.loca.lt
^C
34.6.164.84


In [ ]:
!pip install streamlit scikit-learn pandas numpy
!npm install -g localtunnel



  Using cached streamlit-1.56.0-py3-none-any.whl.metadata (9.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 105.8 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
changed 22 packages in 2s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [ ]:
!pip install streamlit pandas numpy scikit-learn

In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏
changed 22 packages in 1s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧your url is: https://shy-geckos-deny.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.6.164.84:8501



In [ ]:
!curl ipv4.icanhazip.com